<a href="https://colab.research.google.com/github/cemgurbey/plastic-water-segmentation/blob/main/notebooks/plastic_water_segmentation.ipynb" target="_blank">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Plastic Residue Detection on Water Bodies
**Diffusion-synthesized data + SAM 2 masks + YOLO segmentation (PyTorch)**

End-to-end pipeline: generate a synthetic training set, train an instance-segmentation
model, and run inference with a water-contamination metric.

## 1. Setup
Clone the repo and install the package. On Colab, use a GPU runtime
(**Runtime > Change runtime type > GPU**) for the diffusion and training steps.

In [ ]:
!git clone -q https://github.com/cemgurbey/plastic-water-segmentation.git
%cd plastic-water-segmentation
!pip install -q -e ".[notebook]"


## 2. Configuration
All knobs live in `SynthConfig` / `TrainConfig` dataclasses — no scattered globals.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from plastic_water_seg import CATEGORIES, SynthConfig, TrainConfig
from plastic_water_seg.data import (
    DatasetSynthesizer,
    DiffusionInpainter,
    ProceduralPainter,
    generate_water_backgrounds,
)
from plastic_water_seg.eval import contamination_index, predict, visualize
from plastic_water_seg.training import train_yolo

print("Categories:", [c.name for c in CATEGORIES])

synth_cfg = SynthConfig(
    image_size=(512, 512),
    train_count=60,   # bump to 500+ for production runs
    val_count=15,
    min_objects=1,
    max_objects=4,
    seed=42,
)
train_cfg = TrainConfig(model="yolo11m-seg.pt", epochs=10, imgsz=640, batch=8)


## 3. Demo water backgrounds
Procedural water textures so the pipeline runs with zero uploads. Replace
`data/backgrounds/` with real photos (rivers, lakes, canals, sea) when ready.

In [ ]:
bg_dir = Path("data/backgrounds")
paths = generate_water_backgrounds(bg_dir, count=6, size=(640, 640), seed=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, p in zip(axes, paths[:3]):
    ax.imshow(Image.open(p)); ax.axis("off"); ax.set_title(p.name)
plt.tight_layout(); plt.show()


## 4. Synthesize the dataset
**Quickstart (CPU-friendly):** `ProceduralPainter` draws stylized debris directly —
great for smoke-testing the pipeline in seconds.

**Production (GPU):** swap in `DiffusionInpainter`, which inpaints photorealistic
debris with Stable Diffusion and refines each mask with SAM 2.

In [ ]:
# --- Quickstart: procedural painter (runs anywhere) ---
painter = ProceduralPainter()

# --- Production: diffusion inpainting (needs GPU + ~4GB download) ---
# painter = DiffusionInpainter(
#     model_id=synth_cfg.diffusion_model,
#     num_inference_steps=synth_cfg.num_inference_steps,
#     guidance_scale=synth_cfg.guidance_scale,
#     strength=synth_cfg.strength,
# )

synth = DatasetSynthesizer(
    backgrounds_dir=bg_dir,
    output_dir=Path("dataset"),
    painter=painter,
    config=synth_cfg,
)
dataset_yaml = synth.run()
print("Dataset manifest:", dataset_yaml)


## 5. Inspect the generated labels
YOLO segmentation format: `<class_id> x1 y1 x2 y2 ...` (normalized polygons).

In [ ]:
from plastic_water_seg.config import CATEGORY_NAMES

img_dir = Path("dataset/images/train")
lbl_dir = Path("dataset/labels/train")
samples = sorted(img_dir.glob("*.jpg"))[:3]

fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
if len(samples) == 1:
    axes = [axes]
colors = ["#FF3366", "#33CC66", "#3399FF"]

for ax, img_path in zip(axes, samples):
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img); ax.axis("off"); ax.set_title(img_path.name)
    w, h = img.size
    for line in (lbl_dir / f"{img_path.stem}.txt").read_text().splitlines():
        parts = line.split()
        cid, coords = int(parts[0]), np.array(parts[1:], dtype=float).reshape(-1, 2)
        poly = coords * np.array([w, h])
        ax.add_patch(plt.Polygon(poly, closed=True, facecolor=colors[cid % 3],
                                 edgecolor=colors[cid % 3], alpha=0.35, linewidth=2))
        ax.text(poly[0, 0], max(0, poly[0, 1] - 5), CATEGORY_NAMES[cid],
                color="white", fontsize=9, weight="bold",
                bbox=dict(facecolor=colors[cid % 3], alpha=0.85, pad=1, edgecolor="none"))
plt.tight_layout(); plt.show()


## 6. Train YOLO segmentation
Transfer learning from a COCO-pretrained `yolo11m-seg` checkpoint (auto-downloaded).
Checkpoints, metrics and plots land in `logs/yolo11m-seg/`.

In [ ]:
model = train_yolo(dataset_yaml, train_cfg)


## 7. Inference + water contamination index
Run the best checkpoint on an unseen image and compute the
**Surface Plastic Contamination Index** (% of water surface covered by plastic).

In [ ]:
val_images = sorted(Path("dataset/images/val").glob("*.jpg"))
test_path = random.choice(val_images)
print("Test image:", test_path.name)

result = predict("logs/yolo11m-seg/weights/best.pt", test_path, conf=0.35)
print(f"Detected objects: {len(result.class_ids)}")
for name, score in zip(result.class_names, result.scores):
    print(f"  - {name}: {score:.1%}")
print(f"Contamination index: {contamination_index(result):.2f}% of surface")

visualize(result)


## 8. What's next
- Swap procedural backgrounds for real water photos in `data/backgrounds/`.
- Re-run synthesis with `DiffusionInpainter` on a GPU for photorealistic data.
- Scale up (`train_count=1000+`, `epochs=100`) and track mAP in `logs/`.
- Export the best checkpoint to ONNX/TensorRT for edge deployment:
  `yolo export model=logs/yolo11m-seg/weights/best.pt format=onnx`.
